In [ ]:
# -*- coding: utf-8 -*-
"""
ResNet18 + QDA (Per-Class Covariance)
Tiny ImageNet (processed .npy)
20 classes
Train with CE
Test with QDA discriminant
Grid Search
"""

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import random
import zipfile
from copy import deepcopy
from typing import List, Tuple

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Subset, DataLoader, Dataset
from torchvision import transforms


# =============================
# Settings
# =============================

SEED = 42
NUM_WORKERS = 2

MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4

ZIP_PATH = "/content/drive/MyDrive/ML_Project/project_files/First_benchmark/tiny-imagenet-processed.zip"
DATA_ROOT = "/content/drive/MyDrive/ML_Project/data/TINYIMG"

SAVE_DIR = "/content/drive/MyDrive/ML_Project/project_files/QDA_Tiny_Imagenet"
os.makedirs(SAVE_DIR, exist_ok=True)

GN_GROUPS = 32

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


# =============================
# Device
# =============================

def get_best_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


device = get_best_device()
print(f"[INFO] Device: {device}")

PIN_MEM = device.type == "cuda"

if device.type == "cuda":
    torch.backends.cudnn.benchmark = True



# =============================
# GroupNorm
# =============================

def make_gn(C):
    g = min(GN_GROUPS, C)
    while g > 1 and (C % g) != 0:
        g //= 2
    return nn.GroupNorm(max(1, g), C)


# =============================
# ResNet18
# =============================

def conv3x3(in_planes, out_planes, stride=1):
    return nn.Conv2d(
        in_planes,
        out_planes,
        3,
        stride,
        1,
        bias=False
    )


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super().__init__()

        self.conv1 = conv3x3(in_planes, planes, stride)
        self.gn1 = make_gn(planes)

        self.conv2 = conv3x3(planes, planes)
        self.gn2 = make_gn(planes)

        self.shortcut = nn.Sequential()

        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_planes,
                    planes,
                    1,
                    stride,
                    bias=False
                ),
                make_gn(planes)
            )

    def forward(self, x):
        out = torch.relu(self.gn1(self.conv1(x)))
        out = self.gn2(self.conv2(out))
        out += self.shortcut(x)
        return torch.relu(out)


class ResNet18Backbone(nn.Module):
    def __init__(self, nf=64):
        super().__init__()

        self.in_planes = nf
        self.nf = nf

        self.conv1 = conv3x3(3, nf)
        self.gn1 = make_gn(nf)

        self.layer1 = self._make_layer(64, 2, 1)
        self.layer2 = self._make_layer(128, 2, 2)
        self.layer3 = self._make_layer(256, 2, 2)
        self.layer4 = self._make_layer(512, 2, 2)

    def _make_layer(self, planes, blocks, stride):
        layers = []
        layers.append(BasicBlock(self.in_planes, planes, stride))
        self.in_planes = planes

        for _ in range(1, blocks):
            layers.append(BasicBlock(self.in_planes, planes))

        return nn.Sequential(*layers)

    def forward(self, x):
        out = torch.relu(self.gn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)

        out = torch.nn.functional.avg_pool2d(out, out.size(2))
        feat = out.view(out.size(0), -1)
        return feat

    @property
    def out_dim(self):
        return 512


# =============================
# Model
# =============================

class SingleHeadNet(nn.Module):
    def __init__(self, backbone, num_classes=20):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Linear(backbone.out_dim, num_classes)

    def forward(self, x):
        f = self.backbone(x)
        return self.head(f)


# =============================
# Tiny ImageNet Dataset
# =============================

def ensure_extracted(zip_path: str, data_root: str) -> str:
    processed_dir = os.path.join(data_root, "processed")

    if os.path.isdir(processed_dir) and len(os.listdir(processed_dir)) > 0:
        print(f"[INFO] Found processed data at: {processed_dir}")
        return data_root

    if not os.path.isfile(zip_path):
        raise FileNotFoundError(f"ZIP not found at:\n{zip_path}")

    print(f"[INFO] Extracting ZIP from:\n{zip_path}\n-> to:\n{data_root}")
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(data_root)

    processed_dir = os.path.join(data_root, "processed")
    assert os.path.isdir(processed_dir), "processed/ folder missing after unzip!"
    print(f"[INFO] Extracted. processed/ ready at: {processed_dir}")
    return data_root


DATA_ROOT = ensure_extracted(ZIP_PATH, DATA_ROOT)

TIN_IMAGENET_MEAN = (0.4802, 0.4480, 0.3975)
TIN_IMAGENET_STD  = (0.2770, 0.2691, 0.2821)


class TinyImagenet(Dataset):


    def __init__(self, root: str, train: bool = True, transform=None):
        self.root = root
        self.train = train
        self.transform = transform

        split = "train" if self.train else "val"
        xs, ys = [], []

        for num in range(20):
            xs.append(np.load(os.path.join(root, f'processed/x_{split}_{num+1:02d}.npy')))
            ys.append(np.load(os.path.join(root, f'processed/y_{split}_{num+1:02d}.npy')))

        self.data = np.concatenate(np.array(xs))
        self.targets = np.concatenate(np.array(ys)).astype(int)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        img, target = self.data[index], int(self.targets[index])
        img = Image.fromarray(np.uint8(255 * img))
        if self.transform is not None:
            img = self.transform(img)
        return img, target


def get_tiny_imagenet():
    tf_train = transforms.Compose([
        transforms.RandomCrop(64, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(TIN_IMAGENET_MEAN, TIN_IMAGENET_STD)
    ])

    tf_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(TIN_IMAGENET_MEAN, TIN_IMAGENET_STD)
    ])

    train = TinyImagenet(DATA_ROOT, True, tf_train)
    test = TinyImagenet(DATA_ROOT, False, tf_test)

    return train, test


# =============================
# Filter classes + relabel
# =============================

class RelabeledSubset(Subset):
    def __init__(self, dataset, indices, keep_classes):
        super().__init__(dataset, indices)

        mapping = {c: i for i, c in enumerate(sorted(keep_classes))}
        self.targets = [mapping[int(dataset.targets[i])] for i in indices]

    def __getitem__(self, idx):
        x, _ = super().__getitem__(idx)
        return x, self.targets[idx]


def filter_classes(ds, classes):
    idx = [i for i, y in enumerate(ds.targets) if int(y) in classes]
    return RelabeledSubset(ds, idx, classes)


def make_loader(ds, bs, shuffle):
    return DataLoader(
        ds,
        bs,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEM,    persistent_workers=True

    )


# =============================
# QDA Stats
# =============================

@torch.no_grad()
def compute_stats(model, loader, num_classes=20):
    model.eval()

    feats = []
    labels = []

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        f = model.backbone(x)

        feats.append(f.detach().cpu())
        labels.append(y.detach().cpu())

    feats = torch.cat(feats, dim=0)
    labels = torch.cat(labels, dim=0)

    means = []
    inv_covs = []
    logdets = []
    priors = []

    N = feats.size(0)

    for c in range(num_classes):
        cf = feats[labels == c]
        Nc = cf.size(0)

        if Nc == 0:
            raise RuntimeError(f"No samples found for class {c} to compute QDA stats.")

        mean = cf.mean(0)
        centered = cf - mean

        cov = torch.cov(centered.T)
        eps = 1e-5
        cov = cov + eps * torch.eye(cov.size(0))

        sign, ld = torch.slogdet(cov)
        if sign.item() <= 0:
            cov = cov + 1e-3 * torch.eye(cov.size(0))
            sign, ld = torch.slogdet(cov)
            if sign.item() <= 0:
                raise RuntimeError(f"Covariance for class {c} not PD even after regularization.")

        inv = torch.inverse(cov)

        means.append(mean.to(device))
        inv_covs.append(inv.to(device))
        logdets.append(ld.to(device))
        priors.append(torch.tensor(float(Nc) / float(N), device=device))

    means = torch.stack(means, dim=0)
    inv_covs = torch.stack(inv_covs, dim=0)
    logdets = torch.stack(logdets, dim=0)
    priors = torch.stack(priors, dim=0)

    return means, inv_covs, logdets, priors


# =============================
# QDA Evaluation
# =============================

@torch.no_grad()
def eval_qda(model, loader, means, inv_covs, logdets, priors):
    model.eval()

    correct = 0
    total = 0
    K = means.size(0)

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        f = model.backbone(x)

        quads = []
        for k in range(K):
            diff = f - means[k]
            q = torch.sum((diff @ inv_covs[k]) * diff, dim=1)
            quads.append(q.unsqueeze(1))

        quads = torch.cat(quads, dim=1)

        logdets_exp = logdets.unsqueeze(0).expand(quads.size(0), -1)
        priors_log = torch.log(priors.unsqueeze(0).expand(quads.size(0), -1) + 1e-20)

        g = (-0.5 * quads) - 0.5 * logdets_exp + priors_log
        preds = torch.argmax(g, dim=1)

        correct += (preds == y).sum().item()
        total += x.size(0)

    return correct / total


# =============================
# Training
# =============================

def train_one(model, train_loader, test_loader, epochs, lr, num_classes=20):
    crit = nn.CrossEntropyLoss()

    opt = optim.SGD(
        model.parameters(),
        lr=lr,
        momentum=MOMENTUM,
        weight_decay=WEIGHT_DECAY
    )

    best_acc = -1
    best_state = None

    for e in range(1, epochs + 1):
        model.train()

        loss_sum = 0
        corr = 0
        tot = 0

        for x, y in train_loader:
            x = x.to(device)
            y = y.to(device)

            opt.zero_grad()
            out = model(x)
            loss = crit(out, y)
            loss.backward()
            opt.step()

            loss_sum += loss.item() * x.size(0)
            corr += (out.argmax(1) == y).sum().item()
            tot += x.size(0)

        train_acc = corr / tot
        loss_avg = loss_sum / tot

        means, inv_covs, logdets, priors = compute_stats(
            model, train_loader, num_classes=num_classes
        )

        test_acc = eval_qda(
            model, test_loader,
            means, inv_covs, logdets, priors
        )

        if test_acc > best_acc:
            best_acc = test_acc
            best_state = deepcopy(model.state_dict())

        print(
            f"Epoch {e:03d} | "
            f"Loss {loss_avg:.4f} | "
            f"Train {train_acc*100:.2f}% | "
            f"Test(QDA) {test_acc*100:.2f}% | "
            f"Best {best_acc*100:.2f}%"
        )

    return best_acc, best_state


# =============================
# Grid Search
# =============================

def grid_search():
    train, test = get_tiny_imagenet()

    classes = list(range(20))

    train = filter_classes(train, classes)
    test = filter_classes(test, classes)

    print("Train subset:", len(train))
    print("Test subset :", len(test))

    batch_sizes = [32,64]
    epochs_list = [350]
    lrs = [0.01,0.1]

    best_cfg = None
    best_acc = -1
    best_w = None

    init_model = SingleHeadNet(
        ResNet18Backbone(), 20
    )

    init_state = init_model.state_dict()

    for bs in batch_sizes:
        for ep in epochs_list:
            for lr in lrs:
                print(f"\n=== BS={bs} EP={ep} LR={lr} ===")

                model = SingleHeadNet(
                    ResNet18Backbone(), 20
                )

                model.load_state_dict(init_state)
                model.to(device)

                tr_loader = make_loader(train, bs, True)
                te_loader = make_loader(test, bs, False)

                acc, state = train_one(
                    model,
                    tr_loader,
                    te_loader,
                    ep,
                    lr,
                    num_classes=20
                )

                if acc > best_acc:
                    best_acc = acc
                    best_cfg = {
                        "bs": bs,
                        "epochs": ep,
                        "lr": lr
                    }
                    best_w = state

    print("\nBEST:", best_cfg, best_acc)

    path = os.path.join(
        SAVE_DIR,
        "best_tinyimg_qda_20class.pth"
    )

    torch.save({
        "state": best_w,
        "acc": best_acc,
        "cfg": best_cfg
    }, path)

    if best_w is None:
        raise RuntimeError("No best state saved (best_w is None). Check training runs.")

    print("Computing final QDA stats for best model...")

    best_model = SingleHeadNet(ResNet18Backbone(), 20).to(device)
    best_model.load_state_dict(best_w)
    best_model.eval()

    full_loader = make_loader(train, 32, False)

    means, inv_covs, logdets, priors = compute_stats(best_model, full_loader, num_classes=20)

    stats_path = os.path.join(SAVE_DIR, "best_tinyimg_qda_20class_stats.pth")

    torch.save({
        "means": means.cpu(),
        "inv_covs": inv_covs.cpu(),
        "logdets": logdets.cpu(),
        "priors": priors.cpu(),
        "classes": classes
    }, stats_path)

    print("Saved QDA stats ->", stats_path)
    print("Saved model weights ->", path)

    return best_cfg, best_acc, path, stats_path


# =============================
# Main
# =============================

if __name__ == "__main__":
    grid_search()

Mounted at /content/drive
[INFO] Device: cuda
[INFO] Found processed data at: /content/drive/MyDrive/ML_Project/data/TINYIMG/processed
Train subset: 10000
Test subset : 1000

=== BS=32 EP=350 LR=0.01 ===
Epoch 001 | Loss 3.0480 | Train 7.08% | Test(QDA) 25.80% | Best 25.80%
Epoch 002 | Loss 2.7112 | Train 15.04% | Test(QDA) 29.50% | Best 29.50%
Epoch 003 | Loss 2.5515 | Train 20.70% | Test(QDA) 32.70% | Best 32.70%
Epoch 004 | Loss 2.3676 | Train 26.55% | Test(QDA) 36.60% | Best 36.60%
Epoch 005 | Loss 2.2106 | Train 30.88% | Test(QDA) 40.10% | Best 40.10%
Epoch 006 | Loss 2.1241 | Train 33.78% | Test(QDA) 40.50% | Best 40.50%
Epoch 007 | Loss 1.9899 | Train 37.55% | Test(QDA) 40.90% | Best 40.90%
Epoch 008 | Loss 1.9033 | Train 40.45% | Test(QDA) 44.00% | Best 44.00%
Epoch 009 | Loss 1.7823 | Train 43.80% | Test(QDA) 46.90% | Best 46.90%
Epoch 010 | Loss 1.6989 | Train 46.20% | Test(QDA) 47.00% | Best 47.00%
Epoch 011 | Loss 1.6286 | Train 48.31% | Test(QDA) 49.80% | Best 49.80%
Epoch